# 02 - Landing Ingestion
Reads raw JSON from the Volume and copies it verbatim into an immutable Landing zone.
No transformations, no schema changes — a pure, replayable copy of the source.

In [0]:
from pyspark.sql.functions import current_date

SOURCE_PATH = "/Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming"
LANDING_PATH = "/Volumes/sentinel_catalog/sentinel_schema/raw_volume/landing"

raw_df = spark.read.json(SOURCE_PATH)

In [0]:
# Partition by ingestion date so Landing stays organized and re-runnable per day
landing_df = raw_df.withColumn("landing_date", current_date())

(
    landing_df.write
    .mode("append")
    .partitionBy("landing_date")
    .json(LANDING_PATH)
)

print(f"Landed {landing_df.count()} records to {LANDING_PATH}")

Landed 15070 records to /Volumes/sentinel_catalog/sentinel_schema/raw_volume/landing


### Verify: file count, record count, sample, schema

In [0]:
print("File listing:")
display(dbutils.fs.ls(LANDING_PATH))

print("Record count:", spark.read.json(LANDING_PATH).count())

File listing:


path,name,size,modificationTime
dbfs:/Volumes/sentinel_catalog/sentinel_schema/raw_volume/landing/_SUCCESS,_SUCCESS,0,1786208277000
dbfs:/Volumes/sentinel_catalog/sentinel_schema/raw_volume/landing/landing_date=2026-08-08/,landing_date=2026-08-08/,0,1786208282681


Record count: 15070


In [0]:
landing_check_df = spark.read.json(LANDING_PATH)
landing_check_df.printSchema()
display(landing_check_df.limit(5))

root
 |-- amount: string (nullable = true)
 |-- bank_name: string (nullable = true)
 |-- customer_phone: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- receiver_upi_id: string (nullable = true)
 |-- sender_upi_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- landing_date: date (nullable = true)



amount,bank_name,customer_phone,ip_address,receiver_upi_id,sender_upi_id,status,timestamp,transaction_id,landing_date
19014.44,KOTAK,9041311194,186.79.89.170,wspaal190@okaxis,lemvaa35@okicici,SUCCESS,2026-08-01T15:36:11,4e8a195c-291f-40b9-85ef-5d8da93c3ca3,2026-08-08
37682.88,AXIS,9627052188,116.205.114.26,cncvpx554@okaxis,fmngmv874@okhdfc,TIMEOUT,2026-08-01T15:32:32,89123cda-01b4-4f74-85a9-b419bdcacd5a,2026-08-08
25145.98,KOTAK,9911712369,195.122.188.176,zyuhza575@okhdfc,opkvzj838@okicici,TIMEOUT,2026-08-01T15:35:09,4efdfb1a-0f77-4145-bd80-09c800df30f0,2026-08-08
23784.64,PNB,9391332408,202.254.182.11,mrnplm944@okicici,viyqgf546@oksbi,SUCCESS,2026-08-01T15:05:00,27d931f6-1182-4abe-99b2-69270ec4f4a0,2026-08-08
36364.88 INR,PNB,9568692505,158.172.152.110,kznzij544@oksbi,auimuh42@okicici,PENDING,2026-08-01T15:09:51,f3320d40-f0d4-43ed-a72f-64d8fe33b483,2026-08-08
